[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/barmag/fanous-llm-lens/blob/main/notebooks/education/stage2_dash_skip_trigram_reference.ipynb)

In [ ]:
# Setup: install deps + fetch the shared helper on Colab
import sys
if 'google.colab' in sys.modules:
    !pip install -q transformer_lens tokenizers huggingface_hub plotly
    # transformer_lens pulls a newer torch; Colab's pre-installed torchaudio was built
    # against the old torch and its .so then fails to load (undefined symbol:
    # torch_library_impl). transformers imports torchaudio lazily and crashes on it,
    # though nothing here uses audio. Remove it so that import path is skipped.
    !pip uninstall -y -q torchaudio
    !wget -q https://raw.githubusercontent.com/barmag/fanous-llm-lens/main/notebooks/education/tiny.py
import tiny
import torch
import plotly.graph_objects as go

torch.manual_seed(42)
print("device:", tiny.device())

<div dir="rtl">

# المرحلة ٢داش: طبقة انتباه واحدة = خليط من bigram و skip-trigram (على عربي حقيقي)

في المرحلة ٢أ ضفنا كتلة انتباه واحدة وشفنا **شكل** الانتباه. هنا بنوصل للنتيجة الأساسية في ورقة *A Mathematical Framework for Transformer Circuits* (Elhage et al., 2021):

> **محوّل بطبقة واحدة، انتباه بس (من غير MLP)، بيتفكّك بالظبط لمجموع:**
> - **مسار مباشر (bigram):** بيتنبأ من التوكن الحالي بس — من غير سياق.
> - **دوائر skip-trigram (رأس لكل رأس):** بتبص لتوكن أقدم في السياق وتنسخ منه معلومة.

والخليط البسيط ده **معبّر بشكل مفاجئ**. عشان الدوائر تبان **مقروءة** مش ضوضاء، الموديل هنا **مدرّب على نطاق محترم**: ٨ رؤوس، d_model=512، على ~٣٤٠ مليون توكن عربي (فصحى أساسًا + مصري). درّبناه مرة واحدة بسكربت `train_stage2dash.py`؛ النوتبوك ده بيحمّل النتيجة ويفكّكها.

**الطريقة أمينة:** في §٥ ما نختارش ثلاثيات لطيفة بإيدينا — بنبحث بذور تصنيفات **محدّدة مقدّمًا** + بحث غير موجَّه، وكل ثلاثي بيتأكّد على تمريرة محتجوزة (lift > ٠) قبل ما نعدّه. بنعدّ اللي اتأكّد فعلًا، والتصنيف الفاضي أو الضعيف **نتيجة** برضه.

**حدود التجربة:** الكوربَس فصحى-أغلب (المصري نادر)، وبذرة واحدة. التفكيك نفسه **مظبوط رياضيًا** لأي موديل من النوع ده.

</div>

<div dir="ltr">

# Stage 2dash: one attention layer = a bigram + skip-trigram ensemble (on real Arabic)

The central result of *A Mathematical Framework for Transformer Circuits* (Elhage et al., 2021):

> **A one-layer attention-only transformer (no MLP) decomposes *exactly* into a sum of:**
> - **a direct path (bigram):** predicts from the current token alone — no context;
> - **skip-trigram circuits (one per head):** look back at an earlier token and copy from it.

This simple ensemble is **surprisingly expressive**. So the circuits read as signal rather than noise, the model here is trained at a **respectable scale**: 8 heads, d_model=512, on ~340M Arabic tokens (mostly MSA + some Masri). It was trained once by `train_stage2dash.py`; this notebook **loads** the result and decomposes it.

**The method is honest:** §5 does *not* hand-pick pretty triples — it searches **pre-committed
seed categories** plus an unsupervised scan, and every triple is checked on a held-out forward
pass (lift > 0) before it counts. We report *verified counts*, and an empty-or-weak category is
itself a finding.

**Limits:** the corpus is MSA-heavy (Masri is scarce) and it's a single seed. The decomposition itself is **mathematically exact** for any such model.

</div>

<div dir="rtl">

## ١. التفكيك بالظبط · The exact decomposition

مخرجات الموديل (logits) بتساوي بالظبط:

$$\text{logits} = \underbrace{x\,W_U}_{\text{bigram}} \;+\; \underbrace{\sum_{\text{heads, positions}} a_{\text{src}}\;(x_{\text{src}}\,W_V W_O W_U)}_{\text{skip-trigram}}$$

الموديل **من غير LayerNorm** و**shortformer** (الموضع في مسار QK بس)، فالتفكيك يطلع مظبوط بالظبط والمسار المباشر يفضل bigram حقيقي.

</div>

<div dir="ltr">

## 1. The exact decomposition

The model's logits equal, exactly, a **bigram direct path** `x·W_U` plus a sum of per-head
**skip-trigram** terms `aₛ·(xₛ·W_V·W_O·W_U)`. The model is **LayerNorm-free** with a
**shortformer** positional embedding (position in the QK path only), so the split is exact
and the direct path stays a true bigram.

</div>

<div dir="rtl">

## ٢. نحمّل الموديل والمُجزّئ · Load the model & tokenizer

نحمّل نقطة الحفظ المدرّبة (محليًا، وإلا من Hugging Face Hub) — موديل بطبقة واحدة، انتباه بس، LN-free + shortformer، مع BPE عربي صغير (~١٢ ألف). للاختبار السريع في الـ CI نبني نسخة مصغّرة (FORCE_TINY).

</div>

In [ ]:
import os

CKPT_DIR = os.path.join(os.path.dirname(os.path.abspath(tiny.__file__)), "checkpoints", "stage2dash")
HF_REPO = "yassermakram/fanous-stage2dash-attn-only-1l"   # push once from train_stage2dash.py

# A small inline Arabic paragraph used for all eval/forward passes (no network needed,
# in-vocab for the trained model). Masri + MSA mix.
EVAL_TEXT = (
    "القطة بتاكل السمك والولد بيشرب اللبن في البيت. "
    "الجو حلو النهارده واحنا رايحين نتمشى في الشارع. "
    "المدينة كبيرة وفيها ناس كتير بتروح وتيجي كل يوم. "
    "هو قال إنه هيسافر بكرة بدري عشان الشغل المهم."
)

def _model_from_ckpt(ckpt):
    c = ckpt["config"]
    model = tiny.make_tiny_model(
        n_layers=c["n_layers"], n_heads=c["n_heads"], d_vocab=c["d_vocab"],
        n_ctx=c["n_ctx"], d_model=c["d_model"], attn_only=c["attn_only"],
        normalization_type=c["normalization_type"],
        positional_embedding_type=c["positional_embedding_type"])
    model.load_state_dict(ckpt["state_dict"])
    model.eval()
    return model

def _load_real():
    from tokenizers import Tokenizer
    tpath, mpath = os.path.join(CKPT_DIR, "tokenizer.json"), os.path.join(CKPT_DIR, "model.pt")
    if not (os.path.exists(tpath) and os.path.exists(mpath)):
        try:
            from huggingface_hub import hf_hub_download
            tpath = hf_hub_download(HF_REPO, "tokenizer.json")
            mpath = hf_hub_download(HF_REPO, "model.pt")
        except Exception as e:
            raise FileNotFoundError(
                f"No local checkpoint in {CKPT_DIR} and HF repo {HF_REPO!r} is unavailable "
                f"({type(e).__name__}). Either train it once with `python train_stage2dash.py` "
                f"(writes the checkpoint locally), set HF_REPO to a pushed model "
                f"(train_stage2dash.py --push-hub --hf-repo <you>/...), or set FORCE_TINY=True "
                f"for a quick structural demo on a tiny random model.") from e
    ckpt = torch.load(mpath, map_location=tiny.device(), weights_only=False)  # carries a config dict
    return _model_from_ckpt(ckpt), Tokenizer.from_file(tpath)

def _build_tiny():
    # Fast, network-free stand-in for CI: a tiny BPE on EVAL_TEXT + a tiny random model.
    from tokenizers import Tokenizer, models, normalizers, pre_tokenizers, trainers
    tok = Tokenizer(models.BPE(unk_token="[UNK]"))
    tok.normalizer = normalizers.NFKC()
    tok.pre_tokenizer = pre_tokenizers.Whitespace()
    tok.train_from_iterator([EVAL_TEXT] * 20,
                            trainers.BpeTrainer(vocab_size=200, special_tokens=["[UNK]"]))
    model = tiny.make_tiny_model(n_layers=1, n_heads=2, d_vocab=tok.get_vocab_size(),
        n_ctx=32, d_model=64, attn_only=True,
        normalization_type=None, positional_embedding_type="shortformer")
    model.eval()
    return model, tok

def load_model_and_tokenizer():
    return _build_tiny() if globals().get("FORCE_TINY") else _load_real()

model, tok = load_model_and_tokenizer()
VOCAB = tok.get_vocab_size()
id_to_str = {i: (tok.id_to_token(i) or "[?]") for i in range(VOCAB)}
def encode(text):
    return tok.encode(text).ids

print(f"layers=1 heads={model.cfg.n_heads} d_model={model.cfg.d_model} "
      f"n_ctx={model.cfg.n_ctx} vocab={VOCAB}")

<div dir="rtl">

## ٢ب. بذور التصنيفات · Category seeds

عشان ما نختارش الثلاثيات بإيدينا (وندّعي اللي عايزينه)، بنحدّد **مقدّمًا** أربع تصنيفات بذرة بنبحث عنها بشكل منهجي في §٥: تعابير فصحى ثابتة، صيغ دينية، أداة التعريف/الصرف، وتباين فصحى↔مصري. **النسخ الذاتي (self-copy) هو الضابط.** كل ثلاثي بيتأكّد على تمريرة محتجوزة (lift > ٠) قبل ما نعرضه — مش بنعدّ رقم ثابت، بنعدّ اللي اتأكّد فعلًا، والتصنيف الفاضي **نتيجة** برضه.

</div>

<div dir="ltr">

## 2b. Category seeds

To avoid hand-picking triples (and claiming whatever we hoped to find), we fix **four seed
categories up front** and search them systematically in §5: MSA fixed expressions, religious
formulae, the definite article / morphology, and an MSA↔Masri contrast. **Self-copy is the
control.** Every triple is checked on a held-out forward pass (lift > 0) before we show it —
we report *verified counts, not a fixed number*, and an **empty category is itself a finding**.

</div>

In [ ]:
# Category seed lexicons (Arabic-reader curated; edit freely). Self-copy is the control.
# Experiment notebooks can override SEED_LEXICONS (and CKPT_DIR above) before the §5 cells.
SEED_LEXICONS = {
    "MSA fixed expressions": ["الرغم", "بالإضافة", "حين", "بسبب", "أجل"],
    "Religious / formulaic": ["الله", "صلى", "رضي", "شاء", "عليه"],
    "Definite-article / morphology": ["ال", "الذي", "التي"],
    "MSA↔Masri contrast": ["اللي", "عايز", "دلوقتي", "الذي", "يريد", "الآن"],
}
print({k: len(v) for k, v in SEED_LEXICONS.items()})

<div dir="rtl">

## ٣. نثبت التفكيك بإيدينا · The decomposition, by hand

نفكّك الـ logits لتتابع حقيقي لجزئين — **المباشر** (تضمين × `W_U`) و**الـ skip-trigram** (مخرج الانتباه × `W_U`) — ومجموعهم لازم يطابق مخرج الموديل بالظبط: **الفرق ~صفر**.

</div>

In [ ]:
ids = torch.tensor([encode(EVAL_TEXT)[: model.cfg.n_ctx]]).to(tiny.device())
logits, cache = model.run_with_cache(ids)

direct = cache["resid_pre", 0] @ model.W_U + model.b_U   # bigram path       <- (1, ctx, V)
skip   = cache["attn_out", 0] @ model.W_U                # skip-trigram path  <- (1, ctx, V)
print("max |(direct + skip) - logits|:", float((direct + skip - logits).abs().max()))  # ~0

# how big is the skip-trigram contribution relative to the bigram skeleton?
print("||skip|| / ||direct|| (logit space):", round(float(skip.norm() / direct.norm()), 2))

<div dir="rtl">

## ٤. دائرة الـ bigram (المسار المباشر) · The bigram circuit

`W_E W_U` هي **هيكل الـ bigram**: لكل توكن، إيه التوكنز اللي بيرجّحها مباشرة من غير سياق. التوكنز الفصحى المتكررة بتطلع **متلازمات نظيفة** (على الرغم، من أجل، البيت الأبيض). التوكنز المصري (زي "اللي") **ضعيفة** لأن الكوربَس فصحى-أغلب — وده بنفسه ملاحظة عن ندرة بيانات اللهجة.

</div>

<div dir="ltr">

## 4. The bigram circuit

`W_E W_U` is the **bigram skeleton**: for each token, what it directly favours next, with no
context. Frequent MSA tokens give **clean collocations**; Masri tokens (e.g. "اللي") are
**weak** because the corpus is MSA-heavy — itself an observation about dialect-data scarcity.

</div>

In [ ]:
bigram = (model.W_E @ model.W_U).detach().cpu()   # (V, V): current token -> next token

def bigram_top(word, k=6):
    seq = encode(word)
    if not seq:
        return f"{word!r}: empty"
    tid = seq[-1]
    _, idx = torch.topk(bigram[tid], k)
    return f"{id_to_str[tid]:>8} ->  " + " · ".join(id_to_str.get(int(j), "[?]") for j in idx)

print("MSA (well-trained) — crisp collocations:")
for w in ["في", "من", "على", "كان", "البيت"]:
    print(" ", bigram_top(w))
print("\nMasri (data-starved here; corpus is MSA-heavy) — weaker / fragmented:")
for w in ["اللي", "عايز", "دلوقتي"]:
    print(" ", bigram_top(w))

<div dir="rtl">

## ٥. دوائر الـ skip-trigram — الطريقة الأمينة · The honest method

كل واحد من الـ ٨ رؤوس = جدول skip-trigram من دائرتين:
- **QK** = `W_E W_Q W_K^T W_E^T`: لتوكن وجهة، **هنبص على مين** (الـ source)؟
- **OV** = `W_E W_V W_O W_U`: الـ source ده، **إيه الناتج اللي بيرجّحه**؟

الطريقة الأمينة من ٣ خطوات: **(١) تشخيص** — أي رأس بيبص بالمحتوى أصلًا (رأس موضعي مايقدرش يستضيف skip-trigram محتوى)؛ **(٢) تجميع مرشّحين** من بذور التصنيفات + بحث غير موجَّه، مرتّبين بـ `OV·QK`؛ **(٣) تأكيد محتجوز**: نبني `[source … source … dest]` ونشوف هل الموديل الكامل بيرفع `P(output)` فوق خط الـ bigram (lift > ٠). بنعرض **اللي اتأكّد بس**.

**حدود مقدّمًا:** التصنيف بيتعمل على **جملة تشخيص واحدة**؛ والـ BPE بيكسّر البذور (عايز→عا، الذي بتتقسم) فبعض التصنيفات بتطلع ضعيفة. وفي **عيب بنيوي** (§٥ج): رأس واحد مايقدرش يشترط على الـ source والـ dest مع بعض.

</div>

<div dir="ltr">

## 5. The skip-trigram circuits — the honest method

Each of the 8 heads is a skip-trigram table from two circuits: **QK** (`W_E W_Q W_Kᵀ W_Eᵀ`,
which source to attend to) and **OV** (`W_E W_V W_O W_U`, what the attended source promotes).

The honest pipeline is three steps: **(1) diagnose** which heads attend by *content* at all (a
positional head — BOS or previous-token — can't host a content skip-trigram); **(2) pool
candidates** from the seed categories *and* an unsupervised scan, ranked by `OV·QK`; **(3)
verify held-out**: build `[source … source … dest]` and check the full model raises
`P(output)` above the bigram baseline (lift > 0). We print **only verified triples**.

**Limits up front:** the head-kind label is from **one diagnosis sentence**; BPE fragments the
seeds (عايز→عا, الذي splits), so some categories come back weak/empty. And a **structural bug**
(§5c) bounds it: a single head cannot jointly condition on *both* source and destination.

</div>

In [ ]:
import skip_trigrams as st

# §5a — diagnose each head on ONE sentence. (bos/prev are the raw masses the classifier
# thresholds at 0.5; printing them keeps borderline heads honest.)
sent = "هو قال إنه هيسافر بكرة بدري عشان الشغل المهم"
sids = encode(sent)[: model.cfg.n_ctx]
_, hcache = model.run_with_cache(torch.tensor([sids]).to(tiny.device()))
print("Per-head attention kind (positional heads can't host content skip-trigrams):")
for h in range(model.cfg.n_heads):
    patt = hcache["pattern", 0][0, h].detach().cpu()
    n = patt.shape[0]
    den = max(n - 1, 1)
    bos = sum(float(patt[i, 0]) for i in range(1, n)) / den
    prev = sum(float(patt[i, i - 1]) for i in range(1, n)) / den
    print(f"  head {h}: {st.head_attention_kind(patt):>10}   (bos={bos:.2f}  prev={prev:.2f})")

In [ ]:
# §5b — pool ~100 candidates PER CATEGORY, verify the whole pool held-out, keep survivors.
# Each seed source is expanded to its top-5 promoted outputs x top-4 destinations (5 seeds ->
# ~100 candidates), pooled across every content head, deduped, then EVERY candidate is verified
# (not just a top slice) so "most representative" is honestly best-of-~100. Head 0 alone is
# copy-dominated; we scan all content heads and tag the head each survivor lives on.
import skip_trigrams as st

FREQ = min(2500, VOCAB)
CONTENT_HEADS = [h for h in range(model.cfg.n_heads)
                 if st.head_attention_kind(
                     hcache["pattern", 0][0, h].detach().cpu()) == "content"] or [0]

def pool_for(seed_words, heads, want=100):
    src = set(st.seed_ids(encode, id_to_str, seed_words, FREQ)) if seed_words else None
    per_out, per_dst = (5, 4) if seed_words else (1, 1)   # seeds expand; unsupervised scans broad
    cands = []
    for h in heads:
        for v in st.candidate_pool(model, head=h, freq=FREQ, top_n=want, sources=src,
                                   per_source_outputs=per_out, per_source_dests=per_dst):
            cands.append({**v, "head": h})
    cands = st.dedup_triples(cands)                       # same triple from two heads -> one
    cands.sort(key=lambda c: c["score"], reverse=True)
    cands = cands[:want]                                  # the ~100 we pick from
    verified = [st.verify_triple(model, c) for c in cands]
    verified.sort(key=lambda v: v["lift"], reverse=True)
    return verified

def show(label, verified, k=6):
    real = [v for v in verified if v["verified"]]
    strong = [v for v in real if v["lift"] >= 0.05]
    print(f"\n{label}: {len(real)} verified ({len(strong)} with lift>=0.05) of {len(verified)} checked")
    for v in real[:k]:
        s, d, o = id_to_str.get(v['source']), id_to_str.get(v['dest']), id_to_str.get(v['output'])
        print(f"   h{v['head']} [{s} ... {d} -> {o}]   lift={v['lift']:+.3f}  score={v['score']:.1f}")

ALL_VERIFIED = []
for label, seeds in SEED_LEXICONS.items():
    v = pool_for(seeds, CONTENT_HEADS)
    for r in v:
        r["group"] = label
    show(label, v)
    ALL_VERIFIED += [r for r in v if r["verified"]]

show("Self-copy baseline (control . head %d)" % CONTENT_HEADS[0],
     [{**v, "head": CONTENT_HEADS[0]} for v in st.verify_pool(
         model, st.candidate_pool(model, head=CONTENT_HEADS[0], freq=FREQ,
                                  include_self_copy=True, top_n=20))])

unsup = pool_for(None, CONTENT_HEADS)
for r in unsup:
    r["group"] = "Unsupervised"
show("Unsupervised - what else is in here", unsup)
ALL_VERIFIED += [r for r in unsup if r["verified"]]

# Human-readable glosses for the collocations we recognise (others stay un-glossed, honestly).
# Edit freely; an experiment notebook on a different tokenizer will surface different tokens.
GLOSS = {
    ("الرغم", "أن"): "على الرغم من أن — concessive frame",
    ("تبلغ", "العمر"): "تبلغ من العمر — 'is X years old'",
    ("بالإضافة", "ذلك"): "بالإضافة إلى ذلك — 'in addition'",
    ("بسبب", "الحرب"): "بسبب … الحرب — 'because of the war'",
    ("رض", "الله"): "رضي الله — religious formula (BPE split رضي→رض+ي)",
    ("التي", "ها"): "التي … ها — relative pronoun + attached clitic",
    ("الذي", "فيه"): "الذي … فيه — relative pronoun + attached clitic",
}
def glossed(v):
    return GLOSS.get((id_to_str.get(v["source"]), id_to_str.get(v["output"])))

# Most representative per category = highest-lift triple we can gloss (a fragment-completion can
# out-lift a real collocation, so prefer the legible one). Null categories stay visible (§7).
featured, NULLS = [], []
for label in list(SEED_LEXICONS) + ["Unsupervised"]:
    vs = [v for v in ALL_VERIFIED if v.get("group") == label]
    g = [v for v in vs if glossed(v)]
    if vs:
        featured.append(max(g or vs, key=lambda v: v["lift"]))
    else:
        NULLS.append(label)
BOARD = sorted(ALL_VERIFIED, key=lambda v: v["lift"], reverse=True)[:12]
print(f"\nALL_VERIFIED: {len(ALL_VERIFIED)} real triples across "
      f"{len({r['group'] for r in ALL_VERIFIED})} groups; null categories: {NULLS or 'none'}")

<div dir="rtl">

### ٥ج. العيب البنيوي · The structural bug

الثلاثيات اللي اتأكّدت فوق **حقيقية** — بس كل واحد منها هو **أحسن شريحة** للرأس، مش اشتراط ثلاثي كامل. السبب: الـ `OV[source]` **مستقل عن الوجهة** — الوجهة بتقرر *هل* هتبص على الـ source، مش *إيه* اللي هيترفع. يبقى أي source قوي بيرفع **أكتر من ناتج** مهما كانت الوجهة. ده العيب الكلاسيكي بتاع الـ skip-trigram في *Elhage et al. (2021)*: `keep…in→mind` بيجرّ معاه `keep…in→bay`.

</div>

<div dir="ltr">

### 5c. The structural bug

The verified triples above are **real** — but each is the head's **best slice**, not full
three-way conditioning. Reason: `OV[source]` is **destination-independent** — the destination
decides *whether* you attend to the source, not *what* gets promoted. So a strong source
raises **several outputs** regardless of destination. This is the classic skip-trigram bug of
*Elhage et al. (2021)*: wanting `keep…in→mind` forces `keep…in→bay` along with it.

</div>

In [ ]:
# §5c — show the bug as a table: the strongest LEGIBLE verified source promotes a SECOND output
# that rides along no matter the destination. OV[source] is destination-independent, so wanting
# [.. -> output] drags [.. -> other] with it. This is the Arabic keep...in->mind / keep...in->bay.
from IPython.display import HTML, display

ref = max((v for v in featured if glossed(v)), key=lambda v: v["lift"], default=None)
if ref is not None:
    _, OV = st.head_circuits(model, ref["head"])
    s = ref["source"]
    extra = [int(i) for i in torch.topk(OV[s, :FREQ], 4).indices
             if int(i) not in (s, ref["output"])]
    o_bug = extra[0] if extra else ref["output"]
    bug_rows = [
        {"source": s, "dest": ref["dest"], "output": ref["output"], "head": ref["head"],
         "group": "اللي عايزينه · wanted", "gloss": glossed(ref)},
        {"source": s, "dest": ref["dest"], "output": o_bug, "head": ref["head"],
         "group": "بيجي معاه · rides along", "gloss": "destination can't suppress it — the bug"},
    ]
    display(HTML(st.triple_table_html(
        bug_rows, id_to_str,
        title="العيب البنيوي · the structural skip-trigram bug — one source, two outputs",
        note="نفس المصدر بيرفع الاتنين مهما كانت الوجهة · same source raises both regardless "
             "of destination (keep...in->mind forces keep...in->bay)")))
else:
    print("no glossable verified triple to illustrate the bug on (tiny/CI model).")

<div dir="rtl">

## ٦. خريطة الانتباه · Attention heatmap

دائرة الـ QK بتقرر نمط الانتباه. نشوفه لرأس واحد على جملة حقيقية: كل صف (وجهة) بيوزّع انتباهه على التوكنز اللي قبله. (بنعرض من اليمين للشمال زي العربي.)

</div>

In [ ]:
sent = "هو قال إنه هيسافر بكرة بدري عشان الشغل"
sids = encode(sent)[: model.cfg.n_ctx]
str_toks = [id_to_str.get(i, "[?]") for i in sids]
_, hcache = model.run_with_cache(torch.tensor([sids]).to(tiny.device()))
pattern = hcache["pattern", 0][0, 0].detach().cpu().numpy()  # head 0  <- (seq, seq)

fig = go.Figure(go.Heatmap(z=pattern, x=str_toks, y=str_toks, colorscale="Blues"))
fig.update_layout(title="Head 0 attention — " + sent, xaxis=dict(side="top"), height=460, width=560)
fig.update_xaxes(autorange="reversed")  # RTL
fig.update_yaxes(autorange="reversed")
fig.show()

<div dir="rtl">

## ٧. معبّر بشكل مفاجئ · Surprisingly expressive

دي الخلاصة (punchline). من ~١٠٠ مرشّح لكل تصنيف، بنعرض **الأكثر تمثيلًا اللي اتأكّد فعلًا** على شكل جدول زي ورقة *Elhage et al.*: `[source … dest] → output` مع الرفع (lift)، والرأس اللي عايش عليه، وتفسير لغوي بسطر. التفسير ده هو اللي يخلّيه «معبّر»: كل سطر قالب عربي حقيقي اتعلمه الموديل، مش رقم مجمّع.

الجدول التاني بيعرض **كل اللي اتأكّد** (أعلى ١٢ بالرفع) عشان تبان إنها مبعثرة على رؤوس مختلفة. والتصنيفات الفاضية بتفضل ظاهرة كنتيجة سالبة أمينة. الرقم المجمّع (قد إيه السياق بيحرّك التوزيع) اتنقل لحاشية تحت الجدول — دليل، مش العنوان.

</div>

<div dir="ltr">

## 7. Surprisingly expressive

This is the punchline. Out of ~100 candidates per category we surface the **most
representative *verified* triple** as a table in the form of *Elhage et al.*:
`[source … dest] → output` with its held-out lift, the head it lives on, and a one-line
gloss of the Arabic collocation. That gloss is what makes it *expressive* — every row is a
real template the model learned, not an aggregate number.

The second table shows **every verified triple** (top 12 by lift) so you can see they're
scattered across heads. Null categories stay visible as an honest negative. The aggregate
"how much does context move things" number is demoted to a footnote under the board.

</div>

In [ ]:
# §7 — the punchline as paper-style tables: most representative verified triple per category,
# then the full verified board. The aggregate context-shift is demoted to a footnote.
from IPython.display import HTML, display

def with_gloss(v):
    return {**v, "gloss": glossed(v) or ""}

# aggregate (footnote, not headline): how far does the skip-trigram move the bigram distribution?
W = min(16, model.cfg.n_ctx)
flat = encode(EVAL_TEXT)
wins = [flat[i:i + W] for i in range(0, len(flat) - W, 4)][:32]
batch = torch.tensor(wins).to(tiny.device())
with torch.no_grad():
    logits2, cache2 = model.run_with_cache(batch)
    direct2 = cache2["resid_pre", 0] @ model.W_U + model.b_U
    p_full, p_dir = torch.softmax(logits2, -1), torch.softmax(direct2, -1)
reshape = float((p_full - p_dir).abs().max(-1).values.mean())
changed = float((p_full.argmax(-1) != p_dir.argmax(-1)).float().mean())

null_note = ("تصنيفات بلا تأكيد · null categories: " + "، ".join(NULLS)) if NULLS else \
    "كل التصنيفات طلّعت ثلاثيات اتأكّدت · every category produced verified triples"
display(HTML(st.triple_table_html(
    [with_gloss(v) for v in featured], id_to_str,
    title="الأكثر تمثيلًا لكل تصنيف · most representative verified triple per category",
    note="من ~١٠٠ مرشّح/تصنيف بعد التأكيد المحتجوز (lift>٠) · best-of-~100 after held-out "
         "verification.  " + null_note)))
display(HTML(st.triple_table_html(
    [with_gloss(v) for v in BOARD], id_to_str,
    title="كل اللي اتأكّد — أعلى ١٢ بالرفع · the full verified board (top 12 by lift)",
    note=f"لاحظ إن أعلى رفع غالبًا الموديل بيكمّل كلمة BPE متكسّرة (الاكت→ابات) · the top lifts are "
         f"often the model completing a BPE-fragmented word.  السياق بيحرّك التوزيع بمتوسط "
         f"{reshape:.3f} ويقلب التوكن رقم ١ في {changed * 100:.0f}% · context reshapes the dist by "
         f"avg {reshape:.3f}, flips top-1 at {changed * 100:.0f}% of positions")))

<div dir="rtl">

## ٨. الخلاصة · Recap (اللي طلع فعلًا)

- **التفكيك مظبوط:** الـ bigram + skip-trigram = مخرج الموديل بالظبط (الفرق ~١e‑٥).
- **التشخيص:** على جملة التشخيص الواحدة، كل الـ ٨ رؤوس طلعت **content**. التصنيف مربوط بالجملة دي.
- **الطريقة بتطلّع skip-trigram محتوى حقيقية — وكل تصنيف طلّع عشرات الثلاثيات المتأكّدة** (من ١٠٠ مرشّح: ٤٨ تعابير فصحى، ٥٢ دينية، ٤٩ أداة تعريف، ٤٩ تباين، ٧٥ غير موجَّه). الأكثر تمثيلًا (اللي نقدر نفسّره) لكل تصنيف:
  - تعابير فصحى: **الرغم … من → أن** (lift +٠.٢٧، رأس ٦) = «على الرغم من أن».
  - دينية: **رض … ي → الله** (lift +٠.٥٥، رأس ٢) = «رضي الله» (الـ BPE قسّم رضي→رض+ي).
  - أداة تعريف/صرف: **التي … → ها** (lift +٠.٢٩، رأس ١) = اسم موصول + ضمير متصل.
  - تباين فصحى↔مصري: **الذي … → فيه** (lift +٠.١١، رأس ١) — بس الجانب الفصيح بس.
  - غير موجَّه (الأقوى): **تبلغ … من → العمر** (lift +٠.٩٨، رأس ٦) = «تبلغ من العمر».
- **تصحيح لنتيجة قديمة:** تصنيف «أداة التعريف/الصرف» **مش فاضي** زي ما قلنا قبل كده — مع ١٠٠ مرشّح طلّع تطابق الاسم الموصول + الضمير (التي…ها). إنما **التباين اللهجي لسه غايب**: التصنيف طلّع الفصحى (الذي) بس، مفيش «اللي» المصري — فالإشارة اللهجية ضعيفة، متسق مع كوربَس فصحى-أغلب.
- **أعلى رفع = إكمال كلمة متكسّرة:** أقوى الثلاثيات (الاكت→ابات +٠.٩٨، الإ→فاقية +٠.٨٩) هي الموديل بيكمّل كلمة فصحى طويلة قسّمها الـ BPE — نتيجة عن التوكنايزر، مش قالب بين-كلمات. أنضف قالب بين-كلمات = تبلغ…من→العمر.
- **اختيار الرأس بيفرق:** التعابير الحقيقية عايشة على رؤوس ١/٢/٦؛ الرأس ٠ لوحده **مهيمن عليه النسخ** (الضابط: هوكي→هوكي +٠.٩٣، سانت→سانت +٠.٨٩). لو خدت أول رأس بس هتفوّتها.
- **العيب البنيوي بيحدّ السقف:** المصدر القوي بيرفع أكتر من ناتج بصرف النظر عن الوجهة (تبلغ → العمر + أمريكية). فكل ثلاثي اتأكّد هو **أحسن شريحة** للرأس، مش اشتراط ثلاثي كامل.
- **مجمّع:** الـ skip-trigram بيعيد تشكيل التوزيع بمتوسط ٠.١٢ وبيقلب التوكن رقم ١ في ٨٩% من المواضع — السياق كله ساكن في الحد ده.

الخلاصة الأمينة: الموديل بيستضيف skip-trigram محتوى **مقروءة وحقيقية** للفصحى (تعابير، صيغ دينية، تطابق صرفي) مبعثرة على رؤوس مختلفة ومحدودة بالعيب البنيوي — والإشارة اللهجية لسه ضعيفة.

</div>

<div dir="ltr">

## 8. Recap (what actually came out)

- **Decomposition is exact:** bigram + skip-trigram reproduces the model logits (diff ≈ 1e‑5).
- **Diagnosis:** on the single diagnosis sentence all 8 heads classify as **content**. This label
  is tied to that one sentence.
- **The method recovers genuine content skip-trigrams — and every category yields dozens of
  verified triples** (of 100 candidates: 48 MSA-fixed, 52 religious, 49 definite-article, 49
  contrast, 75 unsupervised). The most representative (glossable) per category:
  - MSA fixed: **الرغم … من → أن** (+0.27, head 6) = *على الرغم من أن* ("despite that").
  - Religious: **رض … ي → الله** (+0.55, head 2) = *رضي الله* (BPE split رضي→رض+ي).
  - Definite-article / morphology: **التي … → ها** (+0.29, head 1) = relative pronoun + clitic.
  - MSA↔Masri contrast: **الذي … → فيه** (+0.11, head 1) — but only the MSA side.
  - Unsupervised (strongest): **تبلغ … من → العمر** (+0.98, head 6) = "is X years old".
- **Correction to an earlier claim:** the *definite-article / morphology* category is **not
  null** (as the first cut reported) — with 100 candidates it surfaces relative-pronoun + clitic
  agreement (التي…ها). But the **dialect contrast is still absent**: the contrast category
  surfaces only the MSA الذي, never Masri اللي — so the dialect signal stays weak, consistent
  with an MSA-heavy corpus.
- **Top lift = completing a fragmented word:** the strongest triples (الاكت→ابات +0.98,
  الإ→فاقية +0.89) are the model completing a long MSA word that BPE split — a *tokenizer*
  effect, not a cross-word template. The cleanest cross-word idiom is تبلغ…من→العمر.
- **Head choice matters:** the real expressions live on heads 1/2/6; head 0 alone is
  **copy-dominated** (control: هوكي→هوكي +0.93, سانت→سانت +0.89). A naive "first content head"
  pick misses them.
- **The structural bug caps it:** a strong source raises several outputs regardless of
  destination (تبلغ → العمر *and* أمريكية). So each verified triple is the head's **best slice**,
  not full three-way conditioning.
- **Aggregate:** the skip-trigram reshapes the next-token distribution by avg 0.12 and flips the
  top-1 prediction at 89% of positions — all the model's context-sensitivity lives in that term.

Honest bottom line: the model **does** host legible, real MSA skip-trigrams (fixed expressions,
religious formulae, morphological agreement) — scattered across heads and bounded by the
structural bug — while the dialect signal stays weak.

</div>